# 08 Metrics and Accuracy vs Precision

Read every number in a regression or ML output table — what it means, where it comes from, and when it lies.

## Table of Contents
- [Why this notebook matters](#why-this-notebook-matters)
- [How to use this notebook](#how-to-use)
- [Environment bootstrap](#environment-bootstrap)
- [Load sample data](#load-sample-data)
- [Regression metrics: MSE, RMSE, MAE](#regression-metrics)
- [Goodness of fit: R² and adjusted R²](#r-squared)
- [Bias–variance tradeoff (simulation)](#bias-variance)
- [Accuracy vs precision: the dartboard analogy](#accuracy-vs-precision)
- [Classification metrics: confusion matrix, accuracy, precision, recall, F1](#classification-metrics)
- [ROC, AUC, and the threshold problem](#roc-auc)
- [The metric Rosetta Stone (where each number appears)](#rosetta)
- [Reflection](#reflection)

<a id="why-this-notebook-matters"></a>
## Why This Notebook Matters

Every regression output and every ML evaluation report is dense with numbers: F-statistic, p-value, R², RMSE, MAE, accuracy, precision, recall, F1, ROC-AUC. Most students can produce these numbers but cannot *read* them. This notebook is the one to come back to whenever you see a metric you cannot explain in one sentence to a non-technical colleague.

The goal is to map every metric you will encounter in the rest of the curriculum to:
- a one-sentence definition,
- a worked numerical example,
- the exact place it appears in `statsmodels`, `sklearn`, or `xgboost` output.

<a id="how-to-use"></a>
## How To Use This Notebook
- Work top-to-bottom; the simulations build on each other.
- Several code cells contain `TODO` markers — fill them in before moving on.
- After each section, write 2–3 sentences in your own words answering the interpretation prompt.
- All examples use the bundled `data/sample/macro_quarterly_sample.csv` so no API keys are needed.

<a id="environment-bootstrap"></a>
## Environment Bootstrap
Run this cell first. It locates the repo root and adds `src/` to the path.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root. Start Jupyter from the repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SAMPLE_DIR = PROJECT_ROOT / 'data' / 'sample'
print('Repo root:', PROJECT_ROOT)

<a id="load-sample-data"></a>
## Load Sample Data

We will use the same quarterly macro panel that the regression notebooks use. This way every metric you compute here will reappear, with the same vocabulary, in the OLS and ML notebooks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)

df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head()

<a id="regression-metrics"></a>
## Regression Metrics: MSE, RMSE, MAE

### Definitions
Given true values $y_i$ and predictions $\hat y_i$ for $i = 1, \dots, n$:

- **Mean Squared Error (MSE)** = $\frac{1}{n} \sum_{i=1}^n (y_i - \hat y_i)^2$ — average squared residual. Penalizes large errors heavily because of the square. Units are the *square* of the target's units.
- **Root Mean Squared Error (RMSE)** = $\sqrt{\text{MSE}}$ — same units as the target. Easier to read; "on average we are off by RMSE."
- **Mean Absolute Error (MAE)** = $\frac{1}{n} \sum_{i=1}^n |y_i - \hat y_i|$ — average absolute residual. Treats every error linearly; less sensitive to outliers than RMSE.

### When to use which
- Use **RMSE** when large errors are disproportionately bad (e.g., GDP forecasting where missing a recession is much worse than missing by a quarter).
- Use **MAE** when you want a robust, easy-to-explain "typical error" and outliers should not dominate.
- Always report **both** alongside R² — they answer different questions.

### Worked example
We will fit a tiny linear model of unemployment on its own one-quarter lag (a near-trivial baseline) and compute each metric by hand, then verify with `sklearn`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score

panel = df[['UNRATE']].dropna().copy()
panel['UNRATE_lag'] = panel['UNRATE'].shift(1)
panel = panel.dropna()

X = panel[['UNRATE_lag']].to_numpy()
y = panel['UNRATE'].to_numpy()

model = LinearRegression().fit(X, y)
yhat = model.predict(X)
resid = y - yhat

n = len(y)
mse_manual = float(np.mean(resid ** 2))
rmse_manual = float(np.sqrt(mse_manual))
mae_manual = float(np.mean(np.abs(resid)))

print(f'MSE  (manual): {mse_manual:.4f}    sklearn: {mean_squared_error(y, yhat):.4f}')
print(f'RMSE (manual): {rmse_manual:.4f}    sklearn: {root_mean_squared_error(y, yhat):.4f}')
print(f'MAE  (manual): {mae_manual:.4f}    sklearn: {mean_absolute_error(y, yhat):.4f}')

### Your Turn: Why does RMSE punish outliers?

Add a single large outlier to the residuals (say, +5 percentage points) and recompute RMSE and MAE. Notice which one moves more.

In [ ]:
# TODO: append a +5.0 outlier to `resid` (call the result `resid_out`)
# then compute RMSE and MAE on the new residuals.

resid_out = ...  # np.append(resid, 5.0)
rmse_out = ...   # np.sqrt(np.mean(resid_out ** 2))
mae_out = ...    # np.mean(np.abs(resid_out))

# Expected: RMSE should jump much more than MAE.
# print(f'RMSE before outlier: {rmse_manual:.3f}  after: {rmse_out:.3f}')
# print(f'MAE  before outlier: {mae_manual:.3f}  after: {mae_out:.3f}')

<a id="r-squared"></a>
## Goodness of Fit: R² and Adjusted R²

**R²** (coefficient of determination) compares your model to the simplest possible baseline: "always predict the mean of $y$."

$$
R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}
= 1 - \frac{\sum (y_i - \hat y_i)^2}{\sum (y_i - \bar y)^2}
$$

Interpretation: "My model explains **R²** of the variance in $y$ that the mean baseline does not."

**Adjusted R²** penalizes adding regressors that do not help:

$$
\bar R^2 = 1 - (1 - R^2)\,\frac{n - 1}{n - k - 1}
$$

where $k$ is the number of predictors. Adding a useless variable raises $R^2$ but lowers $\bar R^2$. Always prefer adjusted R² for comparing models with different numbers of regressors.

### Watch out
- A high R² in time series is often misleading: trending variables are correlated with each other for trivial reasons. Always look at residuals (next notebook in section 02).
- R² can be **negative** out-of-sample: that means your model is worse than just predicting the mean. This will happen sometimes with overfit ML models on test data.

In [ ]:
ss_res = float(np.sum((y - yhat) ** 2))
ss_tot = float(np.sum((y - y.mean()) ** 2))
r2_manual = 1 - ss_res / ss_tot
k = X.shape[1]
adj_r2 = 1 - (1 - r2_manual) * (n - 1) / (n - k - 1)

print(f'R² (manual):   {r2_manual:.4f}    sklearn: {r2_score(y, yhat):.4f}')
print(f'Adjusted R²:   {adj_r2:.4f}')

<a id="bias-variance"></a>
## Bias–Variance Tradeoff (Simulation)

Models can be wrong in two distinct ways:

- **Bias** — systematic error from being too simple to capture the true relationship (e.g., fitting a straight line to a curved truth). High-bias models *underfit*.
- **Variance** — sensitivity to the specific training data, so estimates jump around when you re-sample (e.g., a deep tree memorizing the training set). High-variance models *overfit*.

Total expected test error decomposes (roughly) into bias² + variance + irreducible noise. The art of modeling is balancing the two.

We will simulate this directly: generate many small samples from a known nonlinear truth, fit (a) a linear model and (b) a high-degree polynomial, and watch how their predictions scatter.

In [ ]:
from numpy.polynomial import polynomial as P

# True relationship: y = sin(x) + small noise
x_grid = np.linspace(0, 2 * np.pi, 200)
y_truth = np.sin(x_grid)

n_per_sample = 25
n_replications = 30
noise_sd = 0.30

preds_linear = np.zeros((n_replications, len(x_grid)))
preds_poly = np.zeros((n_replications, len(x_grid)))

for r in range(n_replications):
    x_train = rng.uniform(0, 2 * np.pi, n_per_sample)
    y_train = np.sin(x_train) + rng.normal(0, noise_sd, n_per_sample)
    # (a) high-bias / low-variance: degree-1 polynomial
    coeffs1 = np.polyfit(x_train, y_train, deg=1)
    preds_linear[r] = np.polyval(coeffs1, x_grid)
    # (b) low-bias / high-variance: degree-12 polynomial
    coeffs12 = np.polyfit(x_train, y_train, deg=12)
    preds_poly[r] = np.polyval(coeffs12, x_grid)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, preds, title in zip(axes, [preds_linear, preds_poly], ['Degree 1 (high bias)', 'Degree 12 (high variance)']):
    for r in range(n_replications):
        ax.plot(x_grid, preds[r], color='steelblue', alpha=0.2, linewidth=1)
    ax.plot(x_grid, y_truth, color='black', linewidth=2, label='Truth')
    ax.plot(x_grid, preds.mean(axis=0), color='crimson', linewidth=2, label='Mean prediction')
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.legend(loc='lower left')
axes[0].set_ylabel('y')
fig.suptitle('Bias–variance tradeoff: 30 fits on 30 random samples')
plt.tight_layout()
plt.show()

### Interpretation Prompt
Look at the two panels above. In the *left* panel, the 30 fits cluster tightly around the same wrong line — that is **bias**: low variance, large systematic gap from the truth. In the *right* panel, each fit chases its own training noise — that is **variance**: low bias on average, but any individual fit could be wildly wrong.

Write 2–3 sentences: which of these failure modes does Random Forest typically reduce, and which does ridge regression typically reduce?

<a id="accuracy-vs-precision"></a>
## Accuracy vs Precision: The Dartboard Analogy

Accuracy and precision sound interchangeable in casual speech but mean different things in statistics:

- **Accuracy** — how close your estimates are to the *true* value, on average. Low bias.
- **Precision** — how close your estimates are to *each other*. Low variance.

An estimator can be one without the other:

| | High accuracy | Low accuracy |
|---|---|---|
| **High precision** | Tight cluster on the bullseye — ideal | Tight cluster off-center — systematically biased |
| **Low precision** | Spread evenly around the bullseye — noisy but unbiased | Spread all over — useless |

Confusing terminology alert: in **classification metrics** (later in this notebook), `precision` means something *different* — it is the fraction of positive predictions that are correct. The dartboard sense above is the older statistics meaning.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
configs = [
    ('High accuracy + High precision', (0.0, 0.0), 0.10),
    ('Low accuracy + High precision',  (0.6, 0.4), 0.10),
    ('High accuracy + Low precision',  (0.0, 0.0), 0.50),
    ('Low accuracy + Low precision',   (0.6, 0.4), 0.50),
]
n_darts = 30
for ax, (title, (mx, my), spread) in zip(axes.flat, configs):
    for radius, color in zip([1.0, 0.7, 0.4, 0.15], ['#ddd', '#bbb', '#888', '#c00']):
        ax.add_patch(plt.Circle((0, 0), radius, color=color, zorder=0))
    xs = rng.normal(mx, spread, n_darts)
    ys = rng.normal(my, spread, n_darts)
    ax.scatter(xs, ys, color='black', s=18, zorder=2)
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle('Accuracy = closeness to truth; Precision = closeness to each other', y=1.02)
plt.tight_layout()
plt.show()

<a id="classification-metrics"></a>
## Classification Metrics: Confusion Matrix, Accuracy, Precision, Recall, F1

For a binary classifier (e.g., "will the next quarter be a recession?"), every prediction falls in one of four cells:

|                  | Predicted Positive | Predicted Negative |
|------------------|--------------------|--------------------|
| **Actual Positive** | True Positive (TP)  | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN)  |

From these four counts come the four most common metrics:

- **Accuracy** = $(TP + TN) / (TP + FP + TN + FN)$ — fraction correct overall. Misleading on imbalanced data: if 5% of quarters are recessions, predicting "no recession" always scores 95% accuracy and is useless.
- **Precision** = $TP / (TP + FP)$ — of the cases I flagged as positive, how many really were? *Cost-sensitive when false positives are expensive (e.g., over-trading).*
- **Recall (Sensitivity)** = $TP / (TP + FN)$ — of the cases that really were positive, how many did I catch? *Cost-sensitive when false negatives are expensive (e.g., missing a recession).*
- **F1** = harmonic mean of precision and recall = $2 \cdot \frac{P \cdot R}{P + R}$ — single number that punishes ignoring either side.

### Why the harmonic mean for F1?
If precision = 1.0 and recall = 0.0, the arithmetic mean is 0.5 (looks decent), but F1 is 0.0 (correctly catastrophic). Harmonic means are dominated by the smaller value — you cannot rescue F1 by being amazing on one side and terrible on the other.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# Synthetic recession-prediction example: imbalanced, 100 quarters, 12% recessions.
n_q = 100
y_true = np.zeros(n_q, dtype=int)
rec_idx = rng.choice(n_q, size=12, replace=False)
y_true[rec_idx] = 1

# A 'lazy' model: always predict no recession.
y_pred_lazy = np.zeros(n_q, dtype=int)

# A 'sensitive' model: predicts recession whenever a noisy index exceeds 1.
score = rng.normal(0, 1, n_q) + 1.5 * y_true  # higher score on real recessions
y_pred_sens = (score > 1.0).astype(int)

for name, y_pred in [('Lazy (always 0)', y_pred_lazy), ('Sensitive', y_pred_sens)]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    print(f'== {name} ==')
    print('Confusion matrix [[TN,FP],[FN,TP]]:')
    print(cm)
    print(f'  accuracy={acc:.3f}  precision={prec:.3f}  recall={rec:.3f}  F1={f1:.3f}')
    print()

### Your Turn: When Accuracy Lies
Notice the *Lazy* model in the cell above: it has high accuracy (because most quarters are not recessions) but precision and recall on the positive class are both 0. Write 2 sentences explaining which metric you would report to a policymaker, and why.

<a id="roc-auc"></a>
## ROC, AUC, and the Threshold Problem

Most classifiers do not output 0/1 — they output a probability or score. You then choose a **threshold**: "flag as recession if score $\ge$ 0.5." Different thresholds trade off precision and recall.

The **ROC curve** plots the *true positive rate* (= recall) against the *false positive rate* across every possible threshold. **AUC** (area under the curve) summarizes the whole tradeoff in one number:

- AUC = 0.5 — useless model (no better than coin flip).
- AUC = 1.0 — perfect ranking.
- AUC = 0.7–0.9 — typical for real macro classifiers.

AUC is *threshold-independent*: it tells you whether the model ranks positives above negatives, regardless of where you draw the cutoff. That makes it the right metric to report when the threshold has not been chosen yet (or will be chosen by a downstream cost analysis).

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Use the (continuous) `score` from the previous cell.
fpr, tpr, thresholds = roc_curve(y_true, score)
auc = roc_auc_score(y_true, score)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f'Model  (AUC = {auc:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Coin flip (AUC = 0.5)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC curve')
ax.legend(loc='lower right')
ax.set_aspect('equal')
plt.show()

<a id="rosetta"></a>
## The Metric Rosetta Stone

Where to find each metric in the rest of the curriculum:

| Metric | What it answers | OLS (`statsmodels`) | sklearn / xgboost |
|---|---|---|---|
| **t-statistic** | Is this single coefficient ≠ 0? | `results.tvalues`, in the summary table next to each coef | n/a directly (use bootstrap) |
| **p-value** | Probability of a t this large under the null | `results.pvalues`, summary `P>\|t\|` column | n/a directly |
| **F-statistic** | Are the regressors *jointly* useful? | `results.fvalue`, summary header line `F-statistic:` | n/a |
| **R²** | Fraction of variance explained vs the mean | `results.rsquared`, summary header `R-squared:` | `r2_score(y, yhat)` |
| **Adjusted R²** | R² with a penalty for extra regressors | `results.rsquared_adj` | n/a (compute manually) |
| **MSE / RMSE** | Squared / root-squared residual size | `results.mse_resid` (residual MSE) | `mean_squared_error`, `root_mean_squared_error` |
| **MAE** | Average absolute residual | n/a in summary | `mean_absolute_error` |
| **Confusion matrix** | TP / FP / TN / FN counts | n/a (regression model) | `confusion_matrix(y_true, y_pred)` |
| **Accuracy** | Fraction of predictions correct | n/a | `accuracy_score` |
| **Precision** | Of predicted positives, fraction truly positive | n/a | `precision_score` |
| **Recall** | Of true positives, fraction we caught | n/a | `recall_score` |
| **F1** | Harmonic mean of precision & recall | n/a | `f1_score` |
| **ROC-AUC** | Threshold-free ranking quality | n/a | `roc_auc_score(y, prob)` |

Bookmark this cell. Whenever an upcoming notebook prints one of these numbers, come back here for the one-line definition.

<a id="reflection"></a>
## Reflection

1. You fit two models. Model A has lower RMSE but lower R² than Model B on the same dataset. Is that possible? Explain.
2. A classifier reports 99% accuracy on a fraud-detection dataset where 0.5% of transactions are fraudulent. Why is accuracy a terrible metric here, and which metrics would you report instead?
3. In your own words, distinguish *accuracy* (dartboard sense) from *precision* (classification metric sense). Give an example sentence where each is used.